# Figure S1I — Distribution of Signed Grade Differences (LLM vs Gold Standard)

Bar chart showing the distribution of signed grade differences (LLM grade − gold standard grade) across matched toxicity comparisons.

**Data sources** (under `figures/figures_data/figure 1/data`):
- `grade_results_84k_FIXED_FP.csv` — LLM grading results (max grade per patient/toxicity)
- `2026May01_merged_ae_with_apr_full.csv` — gold standard with grades

Outputs are written to `figure 1/results/supp/`.


In [ ]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

%matplotlib inline

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
#   figures/
#   ├── figures_data/figure 1/data/   ← inputs (shared OneDrive data dir)
#   └── v1/figure 1/
#       ├── scripts/                  ← this notebook
#       └── results/supp/             ← output PDF + CSV
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

LLM_PATH  = DATA / "grade_results_84k_FIXED_FP.csv"
GOLD_PATH = DATA / "2026May01_merged_ae_with_apr_full.csv"
OUT_PATH  = RESULTS / "supp" / "Grade_Diff_S1I.pdf"
CSV_OUT   = RESULTS / "supp" / "Grade_Diff_S1I_results.csv"

for p in [LLM_PATH, GOLD_PATH]:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Results: {RESULTS / 'supp'}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
TOXICITY_COLS = [
    "adrenal_insufficiency",
    "colitis",
    "hyperthyroidism",
    "hypothyroidism",
    "pneumonitis",
    "liver_toxicity",
]

# Gold-standard string -> canonical column. Matches Performance_Matrix_Fig1B.
TOXICITY_MAP = {
    'pneumonitis': 'pneumonitis',
    'adrenal insufficiency': 'adrenal_insufficiency',
    'adrenal_insufficiency': 'adrenal_insufficiency',
    'liver toxicity': 'liver_toxicity',
    'liver_toxicity': 'liver_toxicity',
    'colitis': 'colitis',
    'hyperthyroidism': 'hyperthyroidism',
    'hypothyroidism': 'hypothyroidism',
}

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
def load_llm_results(path):
    df = pd.read_csv(path)
    df["mrn"] = df["mrn"].astype(str).str.lstrip("'").str.zfill(8)
    missing = [c for c in TOXICITY_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in LLM file: {missing}")
    return df.groupby("mrn")[TOXICITY_COLS].max().reset_index()

def load_gold_standard(path):
    df = pd.read_csv(path, low_memory=False)
    df["MRN"] = df["MRN"].astype(str).str.lstrip("'").str.zfill(8)
    df["Toxicity_norm"] = df["Toxicity"].str.strip().str.lower()
    df["tox_canonical"] = df["Toxicity_norm"].map(TOXICITY_MAP)
    return df[["MRN", "Toxicity_norm", "tox_canonical", "Grade"]]

llm_df = load_llm_results(LLM_PATH)
gold_df = load_gold_standard(GOLD_PATH)
print(f"LLM rows: {len(llm_df):,}  |  Gold rows: {len(gold_df):,}")

# --- diagnostic: what the map keeps vs drops ---
unmapped = (gold_df[gold_df["tox_canonical"].isna()]
            .groupby("Toxicity_norm")["MRN"].nunique().sort_values(ascending=False))
print(f"\nUnmapped gold toxicity strings (top 15 by patient count):")
print(unmapped.head(15))

In [ ]:
gs_subset = gold_df[gold_df["tox_canonical"].notna()].copy()
gs_lookup = gs_subset.groupby(["MRN", "tox_canonical"])["Grade"].max().to_dict()
shared_mrns = set(llm_df["mrn"]).intersection(gs_subset["MRN"])

comparison_rows = []
for _, row in llm_df.iterrows():
    mrn = row["mrn"]
    if mrn not in shared_mrns:
        continue
    for tox in TOXICITY_COLS:
        llm_grade = row[tox]
        if pd.isna(llm_grade) or llm_grade <= 0:
            continue
        gs_grade = gs_lookup.get((mrn, tox))
        if gs_grade is None or pd.isna(gs_grade):
            continue
        comparison_rows.append({
            "mrn": mrn,
            "toxicity": tox,
            "llm_grade": llm_grade,
            "gs_grade": gs_grade,
            "signed_diff": llm_grade - gs_grade,
        })

comparison_df = pd.DataFrame(comparison_rows)
print(f"Gold patients w/ mapped toxicity: {gs_subset['MRN'].nunique():,}")
print(f"Shared with LLM file: {len(shared_mrns):,}")
print(f"Matched comparisons: {len(comparison_df):,} "
      f"across {comparison_df['mrn'].nunique():,} patients")
print(comparison_df.groupby("toxicity").size().sort_values(ascending=False))

In [ ]:
n_patients = comparison_df["mrn"].nunique()
n_comparisons_check = len(comparison_df)
print(f"{n_patients:,} unique patients contributing {n_comparisons_check:,} matched comparisons")
print(f"Mean comparisons per patient: {n_comparisons_check / n_patients:.2f}")
print(comparison_df.groupby("mrn").size().value_counts().sort_index()
      .rename_axis("comparisons_per_patient").to_frame("n_patients"))

## Null model — permutation test for grade precision

Same logic as the label-shuffle null model used for the time-to-event Brier score
(`Time_Dependent_Validation_S1J`): **precision** here is the exact-match rate --
the fraction of matched comparisons where `llm_grade == gs_grade`. The null model
breaks the pairing between predicted and true grades by randomly shuffling the
`llm_grade` values across all matched comparisons (gold-standard grades held fixed),
then recomputes the exact-match rate under that shuffled pairing. Repeating this many
times builds a null distribution representing "the LLM's predicted grade carries no
real relationship to the true grade." The p-value is the fraction of null (shuffled)
precision scores that are at least as good as the real, unshuffled precision --
i.e. how often random guessing (matched to the same grade distribution) would do
as well as or better than the actual LLM/RAG pipeline. This is a single p-value for
the whole chart (all six toxicities combined), matching how the chart itself is one
aggregate distribution rather than per-toxicity panels.

In [ ]:
# ---------------------------------------------------------------------------
# Null model: permutation test for exact-match grade precision
# ---------------------------------------------------------------------------
N_PERMUTE = 5000
RNG_SEED = 0
rng = np.random.default_rng(RNG_SEED)

llm_grades = comparison_df["llm_grade"].to_numpy()
gs_grades = comparison_df["gs_grade"].to_numpy()
n_comparisons = len(comparison_df)

precision = float(np.mean(llm_grades == gs_grades))

# ---- null distribution: shuffle predicted (LLM) grades against true (gold) grades ----
null_precisions = np.empty(N_PERMUTE)
for i in range(N_PERMUTE):
    shuffled = rng.permutation(llm_grades)
    null_precisions[i] = np.mean(shuffled == gs_grades)

# one-sided: fraction of null (shuffled) precision scores at least as good (>=) as the real one
p_value_vs_null = (np.sum(null_precisions >= precision) + 1) / (N_PERMUTE + 1)  # +1 avoids p=0

print(f"Exact-match precision (LLM == gold standard grade): {precision:.4f}")
print(f"Null model mean precision (shuffled, {N_PERMUTE:,} permutations): {null_precisions.mean():.4f}")
print(f"p-value vs. null: {p_value_vs_null:.4f}  (n_comparisons={n_comparisons:,})")


In [ ]:
# ---------------------------------------------------------------------------
# Figure — Distribution of signed grade differences
# ---------------------------------------------------------------------------
if comparison_df.empty:
    raise RuntimeError("No overlapping MRN/toxicity pairs to plot.")

bins = [-3, -2, -1, 0, 1, 2, 3]
counts = comparison_df["signed_diff"].value_counts().reindex(bins, fill_value=0)
percentages = (counts / counts.sum()) * 100

colors = [
    "#1f4e79",
    "#3c78a8",
    "#76a5d3",
    "#bfbfbf",
    "#e79999",
    "#d06060",
    "#8b0000",
]

# Target size: 180.8029pt x 136.4339pt (Illustrator artboard) = 2.5112in x 1.8949in @ 72pt/in
fig, ax = plt.subplots(figsize=(2.5112, 1.8949))
bars = ax.bar(bins, percentages, width=0.7, color=colors, edgecolor="black", linewidth=0.4)

for bar, pct in zip(bars, percentages):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.6,
        f"{pct:.1f}%",
        ha="center", va="bottom", fontsize=5,
    )

ax.set_xlabel("Signed difference\n(LLM grade \u2212 gold standard grade)", fontsize=6, labelpad=2, linespacing=1.0)
ax.set_ylabel("Comparisons (%)")
ax.set_xticks(bins)
ax.set_ylim(0, percentages.max() + 5)

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("bottom", "left"):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)

# Margins matched exactly to the KM panel (Time_Dependent_Validation_S1J) so the two
# panels' plot areas -- and therefore their y-axes -- align when placed side by side.
fig.subplots_adjust(left=0.20, right=0.97, top=0.95, bottom=0.32)
fig.savefig(OUT_PATH, dpi=450)
print(f"Saved: {OUT_PATH.name}")
plt.show()

## Save results CSV

Signed-difference distribution table (matching the chart) plus the exact-match
precision and its single permutation p-value vs. the null model, broadcast onto
every row so both live in one file.

In [ ]:
results_df = pd.DataFrame({
    "signed_diff": bins,
    "count": counts.values,
    "pct_of_comparisons": percentages.values,
})
results_df["n_comparisons"] = n_comparisons
results_df["exact_match_precision"] = precision
results_df["null_mean_precision"] = null_precisions.mean()
results_df["p_value_vs_null"] = p_value_vs_null

CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT.name}")